# Workflow nets and soundness

A workflow net has one source place `i`, one sink place `o`, and every node lies on a path from `i` to `o`. It is **sound** if from the initial marking (one token in `i`) the net can always reach the final marking (one token in `o` only), no token remains elsewhere when `o` is marked, and no transition is dead. `is_sound` decides this on the reachability graph; `is_sound_pm4py` cross-checks with PM4Py.

```mermaid
flowchart LR
    i((i)) --> register[register] --> a((a)) & b((b))
    a --> checkA[checkA] --> a2((a2))
    b --> checkB[checkB] --> b2((b2))
    a2 & b2 --> decide[decide] --> o((o))
```

`xor_and_mismatch` is the classic bug: an exclusive choice (`chooseA` / `chooseB`) followed by an AND-join `join` that waits for both branches.

In [1]:
from petrilab.petri.workflow import (
    is_sound,
    is_sound_pm4py,
    is_workflow_net,
    parallel_review,
    xor_and_mismatch,
)

print(is_workflow_net(parallel_review()), is_workflow_net(xor_and_mismatch()))

True True


Both nets are structurally valid workflow nets. Soundness is a behavioural property and needs more than the structural check.

In [2]:
print("own:", is_sound(parallel_review()))
print("pm4py:", is_sound_pm4py(parallel_review()))

own: True
pm4py: True


/var/home/alefire/Desktop/Projects/Petri_nets/.venv/lib/python3.13/site-packages/pm4py/algo/analysis/woflan/place_invariants/utility.py:163: UserWarning: solution from scipy may be unstable. Please install PuLP (pip install pulp) for fully reliable results.
  warnings.warn(


The parallel review (AND-split, then AND-join) is sound according to both our checker and PM4Py.

In [3]:
n = xor_and_mismatch()
print("own:", is_sound(n))
print("pm4py:", is_sound_pm4py(xor_and_mismatch()))
g = n.reachable()
print("dead markings:", [n.marking_dict(m) for m, s in g.items() if not s])

own: False
pm4py: False
dead markings: [{'i': 0, 'pa': 1, 'pb': 0, 'o': 0}, {'i': 0, 'pa': 0, 'pb': 1, 'o': 0}]


The XOR/AND mismatch is unsound. Choosing A leaves a token in `pa` and `join` waits forever for `pb` (and vice versa): two dead markings that are not the final marking, so the option to complete is violated.

In [4]:
n = parallel_review()
n.add_place("orphan")
print("workflow net:", is_workflow_net(n), "sound:", is_sound(n))

workflow net: False sound: False


A place not connected between `i` and `o` breaks the workflow-net structure, so the net is rejected before any behavioural check.